# Learning General Agent Skills from Public Trajectories

Runs the `agentskill` pipeline from Colab via the CLI, with the heavy LoRA
step on the GPU **used sparingly** (tiny base model, low rank, few steps).
The collect / score / curate / evaluate steps are CPU-only, so most of the
work — including the baseline-vs-learned improvement — needs no GPU.

Runtime: set **Runtime > Change runtime type > T4 GPU** only before the
fine-tune cell; everything above runs on a CPU runtime.

## 1. Clone the repo (git) + install

In [ ]:
!git clone https://github.com/sugeerth/self-learning-llm-training.git
%cd self-learning-llm-training
!git rev-parse --short HEAD
# CPU pipeline needs nothing beyond the stdlib; install only for the GPU step
!pip -q install numpy pytest

## 2. Pipeline on CPU: collect → score → curate → evaluate

`collect` uses synthetic trajectories offline; to use real public runs, wire
a fetcher into `agentskill/sources.py` (SWE-bench tarballs, GAIA/ML-agent
dumps) — see the repo README.

In [ ]:
!python -m agentskill collect --out trajectories.jsonl
!python -m agentskill score  --in trajectories.jsonl --top 8
!python -m agentskill curate --in trajectories.jsonl --out sft.jsonl

## 3. The headline result: baseline vs trajectory-learned

In [ ]:
!python -m agentskill evaluate

## 4. GPU step (sparingly): LoRA distillation of the curated skills

Switch to a **T4 GPU** runtime first. This distills the same curated
trajectories the retrieval agent uses into LoRA weights: tiny base model,
rank 8, ≤60 steps, gradient checkpointing, bf16.

In [ ]:
!pip -q install 'transformers>=4.44' 'peft>=0.12' 'datasets>=2.20' accelerate
!python -m agentskill finetune --sft sft.jsonl --base-model sshleifer/tiny-gpt2 --max-steps 60

## 5. (Optional) commit results back with git

Set your token, then push the SFT dataset / metrics to a branch.

In [ ]:
# from getpass import getpass
# tok = getpass('GitHub token: ')
# !git config user.email you@example.com && git config user.name you
# !python -m agentskill evaluate --json > agentskill_metrics.json
# !git checkout -b colab-agentskill-run
# !git add agentskill_metrics.json && git commit -m 'Colab agentskill run metrics'
# !git push https://$tok@github.com/sugeerth/self-learning-llm-training colab-agentskill-run